In [10]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
import re
import warnings
warnings.filterwarnings('ignore') 

qa_pairs = [
    ("Какое самое быстрое животное на земле?", "Гепард, он развивает скорость до 120 км/ч."), # КОРОТКИЙ
    ("Как ухаживать за домашней кошкой?", "Нужно кормить качественным кормом, менять воду, чистить лоток, играть и регулярно посещать ветеринара. Также важно делать прививки и обрабатывать от паразитов."), # ДЛИННЫЙ
    ("Сколько живет слон?", "Около 60-70 лет."), # ОЧЕНЬ КОРОТКИЙ
    ("Чем питаются панды?", "Панды питаются в основном бамбуком, который составляет 99% их рациона."),
    ("Где живут пингвины?", "В Антарктиде и других холодных регионах Южного полушария."), # КОРОТКИЙ
    ("Почему собаки виляют хвостом?", "Собаки виляют хвостом по разным причинам: когда радуются, возбуждены или хотят играть. Также это способ коммуникации с другими собаками и людьми. Однако, если хвост поджат, это может означать страх."), # ДЛИННЫЙ
    ("Какие животные впадают в спячку?", "Медведи, ежи, сурки и многие другие животные впадают в зимнюю спячку, чтобы пережить холод и недостаток пищи."),
    ("Умеют ли свиньи плавать?", "Да, свиньи отлично плавают."), # ОЧЕНЬ КОРОТКИЙ
    ("Как дельфины общаются между собой?", "Дельфины общаются с помощью эхолокации и различных звуков: свиста, щелчков и скрипов. У каждого дельфина есть свой уникальный 'автограф' — свист, который работает как имя. Они также используют язык тела."), # ДЛИННЫЙ
    ("Самое большое животное в мире?", "Синий кит — самое большое животное на планете, его длина может достигать 30 метров."),
    ("Что едят кролики?", "Траву, сено, овощи и специальный корм."), # КОРОТКИЙ
    ("Как защищаются скунсы?", "Скунсы выделяют очень неприятно пахнущую жидкость из анальных желез. Они предупреждают об этом, поднимая хвост и топая лапами. Если хищник не уходит, скунс разворачивается и разбрызгивает секрет на расстояние до 3 метров."), # ДЛИННЫЙ
    ("Есть ли у змей веки?", "Нет, глаза змей покрыты прозрачной пленкой."), # КОРОТКИЙ
    ("Какое животное называют 'кораблем пустыни'?", "Верблюда, так как он может долго обходиться без воды и пищи, запасая жир в горбах."),
    ("Сколько лет живут черепахи?", "Многие виды черепах живут более 100 лет.")
]


df_qa = pd.DataFrame(qa_pairs, columns=['Вопрос', 'Ответ'])

print("="*80)
print("БАЗА ЗНАНИЙ (ВОПРОС-ОТВЕТ):")
print("="*80)
for i, row in df_qa.iterrows():
    print(f"{i+1}. ВОПРОС: {row['Вопрос']}")
    print(f"   ОТВЕТ:  {row['Ответ']}\n")

БАЗА ЗНАНИЙ (ВОПРОС-ОТВЕТ):
1. ВОПРОС: Какое самое быстрое животное на земле?
   ОТВЕТ:  Гепард, он развивает скорость до 120 км/ч.

2. ВОПРОС: Как ухаживать за домашней кошкой?
   ОТВЕТ:  Нужно кормить качественным кормом, менять воду, чистить лоток, играть и регулярно посещать ветеринара. Также важно делать прививки и обрабатывать от паразитов.

3. ВОПРОС: Сколько живет слон?
   ОТВЕТ:  Около 60-70 лет.

4. ВОПРОС: Чем питаются панды?
   ОТВЕТ:  Панды питаются в основном бамбуком, который составляет 99% их рациона.

5. ВОПРОС: Где живут пингвины?
   ОТВЕТ:  В Антарктиде и других холодных регионах Южного полушария.

6. ВОПРОС: Почему собаки виляют хвостом?
   ОТВЕТ:  Собаки виляют хвостом по разным причинам: когда радуются, возбуждены или хотят играть. Также это способ коммуникации с другими собаками и людьми. Однако, если хвост поджат, это может означать страх.

7. ВОПРОС: Какие животные впадают в спячку?
   ОТВЕТ:  Медведи, ежи, сурки и многие другие животные впадают в зимнюю спячку

In [18]:
import pymorphy3

morph = pymorphy3.MorphAnalyzer()

def preprocess_text(text):
    """Правильная предобработка текста с лемматизацией"""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    stop_words = {'и', 'в', 'не', 'на', 'я', 'быть', 'с', 'у', 'он', 'она', 'они', 'мы', 'вы', 'это', 'как', 'что', 'где', 'когда', 'кто', 'по', 'для', 'из', 'от', 'до', 'за', 'через', 'после', 'перед', 'о', 'об', 'а', 'но', 'или', 'же', 'ли', 'то', 'так', 'вот', 'все', 'весь', 'вся', 'всё', 'каков', 'ваш'}
    
    words = text.split()
    words = [morph.parse(word)[0].normal_form for word in words if word not in stop_words]
    
    return ' '.join(words)

df_qa['Вопрос_обработанный'] = df_qa['Вопрос'].apply(preprocess_text)
df_qa['Ответ_обработанный'] = df_qa['Ответ'].apply(preprocess_text)

print("После предобработки (лемматизации):")
print(df_qa[['Вопрос_обработанный']].head())

После предобработки (лемматизации):
                  Вопрос_обработанный
0  какой самый быстрый животное земля
1            ухаживать домашний кошка
2                   сколько жить слон
3                  чем питаться панда
4                        жить пингвин


In [12]:

vectorizer = CountVectorizer()
X_questions = vectorizer.fit_transform(df_qa['Вопрос_обработанный'])
X_answers = vectorizer.transform(df_qa['Ответ_обработанный'])

print("Матрица Bag of Words для вопросов:")
print("Размерность:", X_questions.shape)
print("Словарь:", vectorizer.get_feature_names_out()[:10], "...")
print("\n")



Матрица Bag of Words для вопросов:
Размерность: (15, 41)
Словарь: ['большой' 'быстрый' 'веко' 'вилять' 'впадать' 'год' 'дельфин' 'домашний'
 'есть' 'животное'] ...




In [13]:
# Шаг 4: Функции для вычисления сходства
def jaccard_similarity(vec1, vec2):
    """Сходство Жаккара"""
    intersection = np.minimum(vec1, vec2).sum()
    union = np.maximum(vec1, vec2).sum()
    if union == 0:
        return 0
    return intersection / union

def cosine_distance(vec1, vec2):
    """Косинусное расстояние (1 - косинусное сходство)"""
    similarity = cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]
    return 1 - similarity

def pearson_correlation(vec1, vec2):
    """Корреляция Пирсона"""
    if np.std(vec1) == 0 or np.std(vec2) == 0:
        return 0
    correlation, _ = pearsonr(vec1, vec2)
    return correlation

def find_best_match(query, df_qa, vectorizer):
    """Поиск наиболее близкого вопроса в базе знаний"""
    query_processed = preprocess_text(query)
    query_vector = vectorizer.transform([query_processed]).toarray()[0]
    
    results = []
    for idx, row in df_qa.iterrows():
        question_vector = X_questions[idx].toarray()[0]
        
        jaccard = jaccard_similarity(query_vector, question_vector)
        cosine_dist = cosine_distance(query_vector, question_vector)
        pearson_corr = pearson_correlation(query_vector, question_vector)
        
        results.append({
            'Индекс': idx,
            'Вопрос': row['Вопрос'],
            'Ответ': row['Ответ'],
            'Жаккар': jaccard,
            'Косинус': cosine_dist,
            'Пирсон': pearson_corr
        })
    
    return pd.DataFrame(results)



In [14]:
# Шаг 5: Подготовка тестовых вопросов

test_questions_with_answers = [
    "Самое быстрое животное?",            
    "Как ухаживать за кошкой?",           
    "Сколько живет слон?",                
    "Что едят панды?",                     
    "Где обитают пингвины?",               
    "Зачем собаки виляют хвостом?",        
    "Какие животные спят зимой?",          
    "Свиньи умеют плавать?",              
    "Как общаются дельфины?",              
    "Самое большое животное?"              
]

test_questions_without_answers = [
    "Как приготовить пиццу?",
    "Какая столица Франции?",
    "Сколько стоит билет на самолет?",
    "Как починить компьютер?",
    "Когда начинается футбольный матч?"
]

In [15]:
results_with_answers = []
for i, question in enumerate(test_questions_with_answers):
    matches = find_best_match(question, df_qa, vectorizer)
    
    best_jaccard = matches.loc[matches['Жаккар'].idxmax()]
    best_cosine = matches.loc[matches['Косинус'].idxmin()] 
    best_pearson = matches.loc[matches['Пирсон'].idxmax()]
    
    correct_answer = df_qa.iloc[i]['Ответ'] if i < len(df_qa) else "Нет в базе"
    
    results_with_answers.append({
        'Вопрос': question,
        'Правильный ответ': correct_answer,
        'Ответ_Жаккар': best_jaccard['Ответ'],
        'Ответ_Косинус': best_cosine['Ответ'],
        'Ответ_Пирсон': best_pearson['Ответ'],
        'Метрика_Жаккар': best_jaccard['Жаккар'],
        'Метрика_Косинус': best_cosine['Косинус'],
        'Метрика_Пирсон': best_pearson['Пирсон']
    })

df_results_with = pd.DataFrame(results_with_answers)

results_without_answers = []
for question in test_questions_without_answers:
    matches = find_best_match(question, df_qa, vectorizer)
    
    best_jaccard = matches.loc[matches['Жаккар'].idxmax()]
    best_cosine = matches.loc[matches['Косинус'].idxmin()]
    best_pearson = matches.loc[matches['Пирсон'].idxmax()]
    
    results_without_answers.append({
        'Вопрос': question,
        'Правильный ответ': "Нет в базе",
        'Ответ_Жаккар': best_jaccard['Ответ'],
        'Ответ_Косинус': best_cosine['Ответ'],
        'Ответ_Пирсон': best_pearson['Ответ'],
        'Метрика_Жаккар': best_jaccard['Жаккар'],
        'Метрика_Косинус': best_cosine['Косинус'],
        'Метрика_Пирсон': best_pearson['Пирсон']
    })

df_results_without = pd.DataFrame(results_without_answers)

In [ ]:
def print_pretty_table(df, title):
    print(f"\n{'='*100}\n{title}\n{'='*100}")
    for i, row in df.iterrows():
        print(f"Вопрос: {row['Вопрос']}")
        print(f"  Правильный ответ: {row['Правильный ответ']}")
        print(f"  [Жаккар]  Ответ: {row['Ответ_Жаккар']} \n            (Метрика: {row['Метрика_Жаккар']:.4f})")
        print(f"  [Косинус] Ответ: {row['Ответ_Косинус']} \n            (Расстояние: {row['Метрика_Косинус']:.4f})")
        print(f"  [Пирсон]  Ответ: {row['Ответ_Пирсон']} \n            (Корреляция: {row['Метрика_Пирсон']:.4f})")
        print("-" * 100)

print_pretty_table(df_results_with, "ТЕСТИРОВАНИЕ НА ВОПРОСАХ С ОТВЕТАМИ В БАЗЕ")
print_pretty_table(df_results_without, "ТЕСТИРОВАНИЕ НА ВОПРОСАХ БЕЗ ОТВЕТОВ В БАЗЕ")


ТЕСТИРОВАНИЕ НА ВОПРОСАХ С ОТВЕТАМИ В БАЗЕ
Вопрос: Самое быстрое животное?
  Правильный ответ: Гепард, он развивает скорость до 120 км/ч.
  [Жаккар]  Ответ: Гепард, он развивает скорость до 120 км/ч. 
            (Метрика: 0.6000)
  [Косинус] Ответ: Гепард, он развивает скорость до 120 км/ч. 
            (Расстояние: 0.2254)
  [Пирсон]  Ответ: Гепард, он развивает скорость до 120 км/ч. 
            (Корреляция: 0.7539)
----------------------------------------------------------------------------------------------------
Вопрос: Как ухаживать за кошкой?
  Правильный ответ: Нужно кормить качественным кормом, менять воду, чистить лоток, играть и регулярно посещать ветеринара. Также важно делать прививки и обрабатывать от паразитов.
  [Жаккар]  Ответ: Нужно кормить качественным кормом, менять воду, чистить лоток, играть и регулярно посещать ветеринара. Также важно делать прививки и обрабатывать от паразитов. 
            (Метрика: 0.6667)
  [Косинус] Ответ: Нужно кормить качественным кормом

In [17]:

# Шаг 7: Анализ и выводы
print("\n" + "=" * 80)
print("АНАЛИЗ РЕЗУЛЬТАТОВ")
print("=" * 80)

# Средние значения метрик для вопросов с ответами
avg_jaccard_with = df_results_with['Метрика_Жаккар'].mean()
avg_cosine_with = df_results_with['Метрика_Косинус'].mean()
avg_pearson_with = df_results_with['Метрика_Пирсон'].mean()

# Средние значения метрик для вопросов без ответов
avg_jaccard_without = df_results_without['Метрика_Жаккар'].mean()
avg_cosine_without = df_results_without['Метрика_Косинус'].mean()
avg_pearson_without = df_results_without['Метрика_Пирсон'].mean()

print(f"\nСредние метрики для вопросов С ответами:")
print(f"  Жаккар: {avg_jaccard_with:.4f}")
print(f"  Косинус (расстояние): {avg_cosine_with:.4f}")
print(f"  Пирсон: {avg_pearson_with:.4f}")

print(f"\nСредние метрики для вопросов БЕЗ ответов:")
print(f"  Жаккар: {avg_jaccard_without:.4f}")
print(f"  Косинус (расстояние): {avg_cosine_without:.4f}")
print(f"  Пирсон: {avg_pearson_without:.4f}")




АНАЛИЗ РЕЗУЛЬТАТОВ

Средние метрики для вопросов С ответами:
  Жаккар: 0.6600
  Косинус (расстояние): 0.2056
  Пирсон: 0.7819

Средние метрики для вопросов БЕЗ ответов:
  Жаккар: 0.1167
  Косинус (расстояние): 0.7845
  Пирсон: 0.2087
